<div style="
    background: #111827;
    color: white;
    padding: 35px;
    border-radius: 12px;
    margin-bottom: 25px;
    box-shadow: 0 4px 12px rgba(0,0,0,0.15);
    font-family: Arial, sans-serif;
">

   <h1 style="
        margin: 0;
        font-size: 2.5rem;
        text-align: center;
        text-transform:uppercase;
   ">
        Mineração de Regras de Associação com Apriori <br> Market Basket Analysis
   </h1>

   <hr style="
        border: none;
        height: 1px;
        background: rgba(255,255,255,0.2);
        margin: 25px 0 10px 0;
   ">
   <div style="
        display: flex;
        flex-direction:column;
        justify-content:center;
        align-items: center;
        flex-wrap: wrap;
        gap: 15px;
        font-size: .9rem;
        color: #94a3b8;
   ">
       <p style="font-weight: 500; font-size:1.2rem; line-height: 1.5; letter-spacing:2px; margin-bottom:-.5rem;">
            Autor: <strong style="color: white; font-weight: 600;">Wellington M. Santos</strong>
       </p>
       <div style="display: flex; gap: 30px; font-weight: 500; justify-content:center">
            <a href="https://www.linkedin.com/in/wellington-moreira-santos"
               target="_blank"
               style="color:#93c5fd; text-decoration:none;">
               LinkedIn
            </a>
            <a href="mailto:wsantos08@hotmail.com"
               style="color:#93c5fd; text-decoration:none;">
               Contato
            </a>
            <a href="https://github.com/esscova"
               target="_blank"
               style="color:#93c5fd; text-decoration:none;">
               GitHub
            </a>
       </div>
   </div>

</div>

A mineração de regras de associação é uma técnica de descoberta de padrões em bases de dados transacionais. O objetivo é identificar conjuntos de itens que ocorrem frequentemente juntos e, a partir disso, construir regras do tipo "quem compra X tende a comprar Y". O exemplo mais citado na literatura é o das fraldas e cervejas: uma rede americana de supermercados descobriu, ao analisar seus registros de venda, que esses dois produtos eram comprados juntos com frequência surpreendente em certos dias da semana. O conhecimento extraído permitiu reorganizar as prateleiras e aumentar as vendas de ambos, sem qualquer campanha adicional.

Neste notebook aplico esse mesmo processo a um dataset real de e-commerce, o **Online Retail II**, disponibilizado pelo UCI Machine Learning Repository. O conjunto contém transações de uma loja britânica de presentes entre 2009 e 2011, com cerca de um milhão de registros. O dataset é rico em ruído e exige um pré-processamento cuidadoso antes de qualquer análise, o que torna o processo KDD (Knowledge Discovery in Databases) o fio condutor natural deste trabalho.

Para a implementação utilizo `pandas` no tratamento e transformação dos dados, `mlxtend` para o algoritmo Apriori e geração das regras, e `matplotlib` com `seaborn` para as visualizações.

**Base de dados:** https://archive.ics.uci.edu/dataset/502/online+retail+ii

O notebook está organizado nas seguintes seções:

1. Imports e Configurações
2. Coleta e Inspeção dos Dados
3. Análise Exploratória
4. Tratamento e Limpeza dos Dados
5. Transformação para o Formato Transacional
6. Mineração com o Algoritmo Apriori
7. Análise das Regras de Associação
8. Conclusão e Considerações Finais
9. Referências


---

## 1. Imports e Configurações

Utilizo `pandas` para manipulação tabular e `numpy` para operações auxiliares. A `mlxtend` fornece o `TransactionEncoder`, para converter listas de transações em matriz binária, além das funções `apriori` e `association_rules` para a etapa de mineração. As bibliotecas `matplotlib` e `seaborn` cuidam das visualizações. O módulo `warnings` é configurado para suprimir mensagens de aviso que não afetam a execução.


In [ ]:
# !pip install mlxtend openpyxl --quiet

In [7]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
import session_info

from mlxtend.frequent_patterns import apriori, association_rules
from mlxtend.preprocessing import TransactionEncoder

warnings.filterwarnings('ignore')

print('- BIBLIOTECAS CARREGADAS -\n')
session_info.show()

- BIBLIOTECAS CARREGADAS -



In [3]:
# configurações de paleta
plt.rcParams.update({
    'figure.facecolor': '#0f172a',
    'axes.facecolor':   '#1e293b',
    'axes.edgecolor':   '#334155',
    'axes.labelcolor':  '#94a3b8',
    'xtick.color':      '#94a3b8',
    'ytick.color':      '#94a3b8',
    'text.color':       '#f1f5f9',
    'grid.color':       '#334155',
    'grid.linestyle':   '--',
    'grid.alpha':       0.5,
})

## 2. Coleta e Inspeção dos Dados

O dataset Online Retail II está disponível no UCI Machine Learning Repository e pode ser baixado diretamente ou via Kaggle. Ele é distribuído em formato `.xlsx` com duas abas, cada uma cobrindo um ano. Aqui carrego apenas a aba referente a 2010-2011, que contém o maior volume de transações.

### 2.1 Carregamento e visão geral
Inicio verificando o tamanho do dataframe e as primeiras linhas para entender o formato dos dados.

In [ ]:
try:
    df = pd.read_excel('online_retail_II.xlsx', sheet_name='Year 2010-2011')
    print('- DADOS CARREGADOS -')
    print(f"Shape: {df.shape}")
except FileNotFoundError:
    print('Arquivo não encontrado, verifique o caminho.')
except Exception as e:
    print(f'Erro: {str(e)}')

- DADOS CARREGADOS -
Shape: (541910, 8)


In [9]:
df.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


O dataframe possui oito colunas. Cada linha representa um item de uma transação, identificado pelo número da fatura (`Invoice`). Um único pedido pode gerar múltiplas linhas, uma por produto.



### 2.2 Tipos de dados e descrição das colunas

Verifico os tipos de cada coluna antes de avançar, pois o dataset mistura colunas numéricas, textuais e de data.


In [10]:

df.info()


<class 'pandas.DataFrame'>
RangeIndex: 541910 entries, 0 to 541909
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   Invoice      541910 non-null  object        
 1   StockCode    541910 non-null  object        
 2   Description  540456 non-null  object        
 3   Quantity     541910 non-null  int64         
 4   InvoiceDate  541910 non-null  datetime64[us]
 5   Price        541910 non-null  float64       
 6   Customer ID  406830 non-null  float64       
 7   Country      541910 non-null  str           
dtypes: datetime64[us](1), float64(2), int64(1), object(3), str(1)
memory usage: 33.1+ MB



As colunas relevantes para este projeto são:

| Coluna | Descrição |
|---|---|
| `Invoice` | Número da fatura. Faturas iniciadas com "C" indicam cancelamento |
| `StockCode` | Código do produto |
| `Description` | Nome do produto |
| `Quantity` | Quantidade comprada (negativa em cancelamentos) |
| `Price` | Preço unitário |
| `Customer ID` | Identificador do cliente |
| `Country` | País do cliente |



### 2.3 Valores nulos

Verifico a proporção de nulos por coluna para antecipar o impacto das etapas de limpeza.


In [11]:

nulos = df.isnull().sum()
pct   = (nulos / len(df) * 100).round(2)
pd.DataFrame({'nulos': nulos, '%': pct})[nulos > 0]


,nulos,%
Description,1454,0.27
Customer ID,135080,24.93




`Customer ID` concentra a maior parte dos nulos. Registros sem identificação de cliente não permitem rastrear o padrão de compra de um mesmo comprador, por isso serão removidos na etapa de tratamento. A coluna `Description` tem uma proporção menor de ausentes, mas também precisará de atenção.



### 2.4 Estatísticas descritivas

Examino as colunas numéricas para identificar valores anômalos antes de qualquer limpeza.


In [12]:
df[['Quantity', 'Price']].describe()

,Quantity,Price
count,541910.000000,541910.000000
mean,9.552234,4.611138
std,218.080957,96.759765
min,-80995.000000,-11062.060000
25%,1.000000,1.250000
50%,3.000000,2.080000
75%,10.000000,4.130000
max,80995.000000,38970.000000




`Quantity` e `Price` apresentam valores mínimos negativos, o que indica cancelamentos e possíveis registros incorretos. Esses registros serão tratados na seção 4. A presença de valores extremos no máximo também sugere outliers pontuais, mas que não comprometem a análise de associação, pois o Apriori opera sobre presença ou ausência de itens, não sobre quantidades.



---
